# Sequence models vs XGBoost on two credit-card fraud datasets

This notebook benchmarks **three temporal deep models** (ported from
`check-lstm-cnn-with-uid-aggregations.ipynb`) against an **XGBoost** baseline on
two datasets, so the four approaches can be compared head-to-head.

**Models**
1. `lstm_bidir` — `LSTMBidirectionalClassifier`: BiLSTM over the raw feature sequence + masked mean/max pool + static (last-row) tower.
2. `cnn_temporal` — `FraudCNNResAttnV2`: **CNN + ResNet1D + self-attention used as the temporal encoder** (Conv1D stem → residual blocks → CLS + stacked transformer block → mean/max pool + static tower).
3. `cnn_feature_lstm` — `CNNRNNHybridClassifier`: **CNN as a per-transaction feature extractor → BiLSTM** over time (CNN slides over the feature axis of each row, LSTM models the sequence).
4. `xgboost` — gradient-boosted trees on the same per-transaction features (the per-entity history aggregations give it temporal context in flat form).

**Datasets**
| tag | source | entity (sequence key) | split |
|---|---|---|---|
| `IBM` | `card_transaction_v1.csv` (IBM TabFormer, 24.4M rows, 0.12% fraud) | `User`+`Card` | time-based 70/15/15 |
| `Sparkov` | `fraudTrain.csv` / `fraudTest.csv` (1.85M rows, ~0.5% fraud) | `cc_num` | natural train/test (val = last 10% of train) |

**Method.** Each transaction becomes a sample whose input is its left-padded
window of the previous `WINDOW` transactions *of the same card* (full history is
preserved; windows are built on-the-fly to avoid materializing a giant
`(N, T, F)` tensor). Training anchors keep **all fraud + a subsampled set of
legitimate** transactions; the **test set is evaluated in full** for honest
ROC-AUC / PR-AUC. Standardizer and frequency encoders are fit on **train rows
only** to avoid leakage.

> Set `SMOKE = False` in the config cell for the full run. `SMOKE = True` runs a
> tiny, fast sanity pass. On Kaggle, attach the two datasets and enable the GPU.

## 0 · Configuration, imports, device, data paths

In [1]:
import os, gc, math, time, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import xgboost as xgb

# ============================ CONFIG ============================
SMOKE         = False     # <<< set True for a fast sanity pass, False for the real run
WINDOW        = 20        # transactions of history per sample
SEED          = 42
SEQ_DTYPE     = np.float16 # window cache dtype (halves memory; cast to float32 in the loop)

# training hyper-parameters
BATCH         = 1024
EPOCHS        = 15
LR            = 1e-3
WEIGHT_DECAY  = 2e-4
GRAD_CLIP     = 1.0
EARLY_STOP    = 5
NEG_PER_POS   = 50        # train/val: keep all fraud + this many legit per fraud
TRAIN_CAP     = None      # optional hard cap on #train anchors (None = no cap)
TEST_CAP      = None      # optional cap on #test anchors (None = full test, honest metrics)
XGB_TREES     = 600
NUM_WORKERS   = 4         # DataLoader workers (use 0 on macOS/MPS)

# which datasets to run
RUN_IBM       = True
RUN_SPARKOV   = True

if SMOKE:
    EPOCHS, BATCH, EARLY_STOP, NEG_PER_POS = 2, 256, 2, 20
    XGB_TREES, NUM_WORKERS = 60, 0
    IBM_NROWS, SPARKOV_NROWS = 150_000, 150_000
else:
    IBM_NROWS, SPARKOV_NROWS = None, None   # None = read full files

# ---- data paths: auto-detect Kaggle vs local; edit the candidate lists if needed ----
def find_first(paths, what):
    for p in paths:
        if p and os.path.exists(p):
            return p
    raise FileNotFoundError(f"Could not locate {what}. Tried:\n  " + "\n  ".join(map(str, paths)))

IBM_CSV = find_first([
    "/kaggle/input/datasets/bachhoviet/graph-lstm-datasets/card_transaction_v1.csv",
    "/kaggle/input/credit-card-transactions/card_transaction_v1.csv",
], "IBM TabFormer card_transaction csv") if RUN_IBM else None

SPARKOV_DIR_CANDIDATES = [
    "/kaggle/input/fraud-detection",
    "/kaggle/input/datasets/bachhoviet/sparkov-card-detection",
]
if RUN_SPARKOV:
    SPARKOV_TRAIN = find_first([f"{d}/fraudTrain.csv" for d in SPARKOV_DIR_CANDIDATES], "Sparkov fraudTrain.csv")
    SPARKOV_TEST  = find_first([f"{d}/fraudTest.csv"  for d in SPARKOV_DIR_CANDIDATES], "Sparkov fraudTest.csv")

def resolve_device():
    if torch.cuda.is_available():        return torch.device("cuda")
    if torch.backends.mps.is_available(): return torch.device("mps")
    return torch.device("cpu")

device = resolve_device()
def set_seed(seed): torch.manual_seed(seed); np.random.seed(seed)
set_seed(SEED)
print(f"torch {torch.__version__} | device={device} | SMOKE={SMOKE} | WINDOW={WINDOW}")

torch 2.10.0+cu128 | device=cuda | SMOKE=False | WINDOW=20


## 1 · Models

All three architectures are copied verbatim from the source notebook so this
file is fully self-contained (no external imports needed on Kaggle).

### 1a · `cnn_temporal` — Conv1D + ResNet1D + self-attention as the temporal encoder

In [2]:
class ResBlock1D(nn.Module):
    def __init__(self, c, k=3, drop=0.1):
        super().__init__()
        p = k // 2
        self.conv1 = nn.Conv1d(c, c, k, padding=p); self.bn1 = nn.BatchNorm1d(c)
        self.conv2 = nn.Conv1d(c, c, k, padding=p); self.bn2 = nn.BatchNorm1d(c)
        self.drop = nn.Dropout(drop); self.act = nn.ReLU()
    def forward(self, x):
        identity = x
        out = self.act(self.bn1(self.conv1(x)))
        out = self.drop(out)
        out = self.bn2(self.conv2(out))
        return self.act(out + identity)

class TransformerBlock(nn.Module):
    def __init__(self, c, n_heads, drop=0.2):
        super().__init__()
        self.attn = nn.MultiheadAttention(c, n_heads, dropout=drop, batch_first=True)
        self.norm1 = nn.LayerNorm(c)
        self.ff = nn.Sequential(nn.Linear(c, c*2), nn.GELU(), nn.Dropout(drop), nn.Linear(c*2, c))
        self.norm2 = nn.LayerNorm(c)
    def forward(self, h, key_padding_mask):
        a, _ = self.attn(h, h, h, key_padding_mask=key_padding_mask, need_weights=False)
        h = self.norm1(h + a)
        h = self.norm2(h + self.ff(h))
        return h

class FraudCNNResAttnV2(nn.Module):
    """CNN + ResNet + stacked self-attention used as the TEMPORAL encoder."""
    def __init__(self, n_features, window=20, c_hidden=128, n_resblocks=2,
                 n_attn_layers=1, n_heads=4, drop=0.2, static_hidden=64,
                 use_cls=True, use_max_pool=True, use_static_tower=True, output_dim=1):
        super().__init__()
        assert c_hidden % n_heads == 0
        self.window=window; self.use_cls=use_cls
        self.use_max_pool=use_max_pool; self.use_static_tower=use_static_tower
        self.stem = nn.Sequential(nn.Conv1d(n_features, c_hidden, 3, padding=1),
                                  nn.BatchNorm1d(c_hidden), nn.ReLU())
        self.resblocks = nn.Sequential(*[ResBlock1D(c_hidden, drop=drop) for _ in range(n_resblocks)])
        self.pos = nn.Parameter(torch.zeros(1, window, c_hidden)); nn.init.trunc_normal_(self.pos, std=0.02)
        if use_cls:
            self.cls = nn.Parameter(torch.zeros(1, 1, c_hidden)); nn.init.trunc_normal_(self.cls, std=0.02)
        self.attn_blocks = nn.ModuleList([TransformerBlock(c_hidden, n_heads, drop) for _ in range(n_attn_layers)])
        if use_static_tower:
            self.static_mlp = nn.Sequential(nn.Linear(n_features, 256), nn.ReLU(), nn.Dropout(drop),
                                            nn.Linear(256, static_hidden), nn.ReLU())
        pool_dim = c_hidden + (c_hidden if use_max_pool else 0) + (c_hidden if use_cls else 0) + (static_hidden if use_static_tower else 0)
        self.head = nn.Sequential(nn.Dropout(drop), nn.Linear(pool_dim, 64), nn.ReLU(),
                                  nn.Dropout(drop), nn.Linear(64, output_dim))
    def forward(self, x, lengths):
        B, T, F = x.shape
        last_row = x[:, -1, :]
        h = self.stem(x.transpose(1, 2))
        h = self.resblocks(h).transpose(1, 2)
        h = h + self.pos
        idx = torch.arange(T, device=h.device).unsqueeze(0)
        real = (T - 1 - idx) < lengths.unsqueeze(1)
        if self.use_cls:
            h = torch.cat([self.cls.expand(B, -1, -1), h], dim=1)
            cls_real = torch.ones(B, 1, dtype=torch.bool, device=h.device)
            key_padding_mask = ~torch.cat([cls_real, real], dim=1)
        else:
            key_padding_mask = ~real
        for blk in self.attn_blocks:
            h = blk(h, key_padding_mask)
        if self.use_cls:
            cls_out = h[:, 0, :]; seq_h = h[:, 1:, :]
        else:
            seq_h = h
        mask_f = real.unsqueeze(-1).float()
        cnt = mask_f.sum(1).clamp(min=1.0)
        parts = [(seq_h * mask_f).sum(1) / cnt]
        if self.use_max_pool:
            parts.append(seq_h.masked_fill(~real.unsqueeze(-1), float("-inf")).max(1).values)
        if self.use_cls:
            parts.append(cls_out)
        if self.use_static_tower:
            parts.append(self.static_mlp(last_row))
        return self.head(torch.cat(parts, dim=1))

### 1b · `lstm_bidir` — BiLSTM + mean/max pool + static tower

In [3]:
class LSTMBidirectionalClassifier(nn.Module):
    def __init__(self, n_features, hidden_dim=128, num_layers=2, drop_prob=0.3,
                 output_dim=1, use_static_tower=True, bidirectional=True):
        super().__init__()
        self.use_static_tower = use_static_tower
        self.lstm = nn.LSTM(n_features, hidden_dim, num_layers, batch_first=True,
                            dropout=drop_prob if num_layers > 1 else 0.0, bidirectional=bidirectional)
        lstm_out = hidden_dim * (2 if bidirectional else 1)
        seq_out = 2 * lstm_out
        if use_static_tower:
            self.static_mlp = nn.Sequential(nn.Linear(n_features, 256), nn.ReLU(), nn.Dropout(drop_prob),
                                            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(drop_prob),
                                            nn.Linear(128, 64), nn.ReLU())
            combined = seq_out + 64
        else:
            combined = seq_out
        self.head = nn.Sequential(nn.Linear(combined, 64), nn.ReLU(), nn.Dropout(drop_prob),
                                  nn.Linear(64, output_dim))
    def forward(self, x, lengths):
        B, T, _ = x.shape
        lstm_out, _ = self.lstm(x)
        idx = torch.arange(T, device=x.device).unsqueeze(0)
        mask = (T - 1 - idx) < lengths.to(x.device).unsqueeze(1)
        mask_f = mask.unsqueeze(-1).float()
        mean_pool = (lstm_out * mask_f).sum(1) / mask_f.sum(1).clamp(min=1.0)
        neg_inf = torch.finfo(lstm_out.dtype).min
        max_pool = lstm_out.masked_fill(~mask.unsqueeze(-1), neg_inf).max(1).values
        seq_vec = torch.cat([mean_pool, max_pool], dim=1)
        if self.use_static_tower:
            seq_vec = torch.cat([seq_vec, self.static_mlp(x[:, -1, :])], dim=1)
        return self.head(seq_vec).squeeze(-1)

### 1c · `cnn_feature_lstm` — per-row CNN feature extractor → BiLSTM

In [4]:
def left_padded_real_mask(lengths, T, device):
    lengths = lengths.to(device=device, dtype=torch.long).clamp(min=1, max=T)
    idx = torch.arange(T, device=device).unsqueeze(0)
    return (T - 1 - idx) < lengths.unsqueeze(1)

def left_to_right_padded(h, lengths):
    B, T, E = h.shape; device = h.device
    lengths = lengths.to(device=device, dtype=torch.long).clamp(min=1, max=T)
    dst = torch.arange(T, device=device).unsqueeze(0).expand(B, T)
    src = (T - lengths.unsqueeze(1) + dst).clamp(min=0, max=T - 1)
    out = h.gather(1, src.unsqueeze(-1).expand(B, T, E))
    real = dst < lengths.unsqueeze(1)
    return out.masked_fill(~real.unsqueeze(-1), 0.0), real

def masked_mean_max(h, real):
    mask_f = real.unsqueeze(-1).float()
    mean = (h * mask_f).sum(1) / mask_f.sum(1).clamp(min=1.0)
    neg_inf = torch.finfo(h.dtype).min
    maxv = h.masked_fill(~real.unsqueeze(-1), neg_inf).max(1).values
    return mean, maxv

class FeatureResBlock1D(nn.Module):
    def __init__(self, c, k=3, drop=0.1):
        super().__init__()
        p = k // 2
        self.conv1 = nn.Conv1d(c, c, k, padding=p); self.bn1 = nn.BatchNorm1d(c)
        self.conv2 = nn.Conv1d(c, c, k, padding=p); self.bn2 = nn.BatchNorm1d(c)
        self.drop = nn.Dropout(drop); self.act = nn.ReLU()
    def forward(self, x):
        identity = x
        out = self.act(self.bn1(self.conv1(x)))
        out = self.drop(out)
        out = self.bn2(self.conv2(out))
        return self.act(out + identity)

class CNNFeatureExtractor(nn.Module):
    """CNN slides over the FEATURE axis of each transaction row -> a per-row embedding.
    Only real (non-padded) rows are processed, then scattered back into place."""
    def __init__(self, n_features, feature_channels=32, embed_dim=128, n_resblocks=2, drop=0.2):
        super().__init__()
        self.n_features = n_features; self.embed_dim = embed_dim
        self.stem = nn.Sequential(nn.Conv1d(1, feature_channels, 5, padding=2),
                                  nn.BatchNorm1d(feature_channels), nn.ReLU())
        self.resblocks = nn.Sequential(*[FeatureResBlock1D(feature_channels, drop=drop) for _ in range(n_resblocks)])
        self.proj = nn.Sequential(nn.Linear(feature_channels * 2, embed_dim), nn.ReLU(),
                                  nn.Dropout(drop), nn.LayerNorm(embed_dim))
    def forward(self, x, real=None):
        B, T, F = x.shape
        flat = x.reshape(B * T, F)
        if real is None:
            row_x = flat.unsqueeze(1).contiguous()
            h = self.resblocks(self.stem(row_x))
            return self.proj(torch.cat([h.mean(2), h.max(2).values], dim=1)).view(B, T, self.embed_dim)
        real_idx = real.reshape(-1).nonzero(as_tuple=False).squeeze(1)
        out = flat.new_zeros(B * T, self.embed_dim)
        if real_idx.numel() == 0:
            return out.view(B, T, self.embed_dim)
        row_x = flat.index_select(0, real_idx).unsqueeze(1).contiguous()
        h = self.resblocks(self.stem(row_x))
        emb = self.proj(torch.cat([h.mean(2), h.max(2).values], dim=1))
        out = out.index_copy(0, real_idx, emb)
        return out.view(B, T, self.embed_dim)

class StaticTower(nn.Module):
    def __init__(self, n_features, static_hidden=64, drop=0.3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_features, 256), nn.ReLU(), nn.Dropout(drop),
                                 nn.Linear(256, 128), nn.ReLU(), nn.Dropout(drop),
                                 nn.Linear(128, static_hidden), nn.ReLU())
    def forward(self, last_row):
        return self.net(last_row)

class CNNRNNHybridClassifier(nn.Module):
    """CNN feature extractor per row -> BiLSTM over time."""
    def __init__(self, n_features, rnn_type="lstm", cnn_embed_dim=128, rnn_hidden=128,
                 rnn_layers=2, bidirectional=True, drop=0.3, use_static_tower=True,
                 use_mean_max_pool=True, use_packed_rnn=True):
        super().__init__()
        self.rnn_type = rnn_type.lower()
        self.use_static_tower = use_static_tower
        self.use_mean_max_pool = use_mean_max_pool
        self.use_packed_rnn = use_packed_rnn
        self.feature_cnn = CNNFeatureExtractor(n_features, embed_dim=cnn_embed_dim, drop=drop)
        rnn_cls = nn.GRU if self.rnn_type == "gru" else nn.LSTM
        self.rnn = rnn_cls(cnn_embed_dim, rnn_hidden, rnn_layers, batch_first=True,
                           dropout=drop if rnn_layers > 1 else 0.0, bidirectional=bidirectional)
        rnn_out = rnn_hidden * (2 if bidirectional else 1)
        pool_dim = rnn_out + (2 * rnn_out if use_mean_max_pool else 0)
        if use_static_tower:
            self.static_mlp = StaticTower(n_features, 64, drop)
            pool_dim += 64
        self.head = nn.Sequential(nn.Linear(pool_dim, 64), nn.ReLU(), nn.Dropout(drop), nn.Linear(64, 1))
    def forward(self, x, lengths):
        B, T, _ = x.shape
        lengths = lengths.to(device=x.device, dtype=torch.long).clamp(min=1, max=T)
        left_real = left_padded_real_mask(lengths, T, x.device)
        last_row = x[:, -1, :]
        h_left = self.feature_cnn(x, real=left_real)
        if self.use_packed_rnn:
            h, rnn_real = left_to_right_padded(h_left, lengths)
            packed = pack_padded_sequence(h, lengths.detach().cpu(), batch_first=True, enforce_sorted=False)
            packed_out, _ = self.rnn(packed)
            rnn_out, _ = pad_packed_sequence(packed_out, batch_first=True, total_length=T)
            rnn_out = rnn_out.masked_fill(~rnn_real.unsqueeze(-1), 0.0)
            last_idx = (lengths - 1).clamp(min=0)
            last_hidden = rnn_out[torch.arange(B, device=x.device), last_idx, :]
        else:
            rnn_out, _ = self.rnn(h_left)
            rnn_real = left_real
            rnn_out = rnn_out.masked_fill(~rnn_real.unsqueeze(-1), 0.0)
            last_hidden = rnn_out[:, -1, :]
        parts = [last_hidden]
        if self.use_mean_max_pool:
            mean, maxv = masked_mean_max(rnn_out, rnn_real)
            parts.extend([mean, maxv])
        if self.use_static_tower:
            parts.append(self.static_mlp(last_row))
        return self.head(torch.cat(parts, dim=1)).squeeze(-1)

def build_model(name, n_features):
    if name == "lstm_bidir":
        return LSTMBidirectionalClassifier(n_features, use_static_tower=True)
    if name == "cnn_temporal":
        return FraudCNNResAttnV2(n_features, window=WINDOW, c_hidden=128, n_resblocks=2,
                                 n_attn_layers=1, n_heads=4)
    if name == "cnn_feature_lstm":
        return CNNRNNHybridClassifier(n_features, rnn_type="lstm")
    raise ValueError(name)

## 2 · On-the-fly windowing `Dataset`

Instead of materializing a `(N, WINDOW, F)` tensor (≈29 GB for the full IBM
set), we keep one flat `(N, F)` float16 matrix sorted by `(entity, time)` and
slice each sample's left-padded window on demand. `block_start[i]` marks where
row `i`'s entity block begins, so a window never bleeds across cards.

In [5]:
class WindowDataset(Dataset):
    def __init__(self, feat, block_start, y, anchors, window):
        self.feat = feat            # (N, F) float16, sorted by (entity, time)
        self.block_start = block_start
        self.y = y
        self.anchors = anchors
        self.window = window
        self.F = feat.shape[1]
    def __len__(self):
        return len(self.anchors)
    def __getitem__(self, j):
        i = int(self.anchors[j])
        start = max(int(self.block_start[i]), i - self.window + 1)
        seq = self.feat[start:i + 1]
        L = seq.shape[0]
        x = np.zeros((self.window, self.F), dtype=np.float32)
        x[self.window - L:] = seq            # LEFT-pad: real steps at the right end
        return torch.from_numpy(x), L, np.float32(self.y[i])

def make_loader(feat, block_start, y, anchors, batch, shuffle, num_workers):
    return DataLoader(WindowDataset(feat, block_start, y, anchors, WINDOW),
                      batch_size=batch, shuffle=shuffle, num_workers=num_workers,
                      pin_memory=False, drop_last=False)

## 3 · Training & evaluation

In [6]:
def evaluate(model, loader, device):
    model.eval(); preds = []
    with torch.no_grad():
        for xb, lb, _ in loader:
            xb = xb.to(device=device, dtype=torch.float32)
            lb = lb.to(device=device, dtype=torch.long)
            preds.append(torch.sigmoid(model(xb, lb).squeeze(-1)).float().cpu().numpy())
    return np.concatenate(preds)

def train_deep(name, n_features, feat, block_start, y,
               tr_anchors, va_anchors, te_anchors, device,
               epochs, batch, lr, weight_decay, grad_clip, early_stop, seed,
               num_workers, verbose=True):
    set_seed(seed)
    model = build_model(name, n_features).to(device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    loss_fn = nn.BCEWithLogitsLoss()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    tr_loader = make_loader(feat, block_start, y, tr_anchors, batch, True,  num_workers)
    va_loader = make_loader(feat, block_start, y, va_anchors, batch, False, num_workers)
    te_loader = make_loader(feat, block_start, y, te_anchors, batch, False, num_workers)
    steps = max(1, math.ceil(len(tr_anchors) / batch))
    sched = None
    if epochs * steps >= 20:    # OneCycleLR needs enough total steps for its phases
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, epochs=epochs,
                    steps_per_epoch=steps, pct_start=0.1, anneal_strategy="cos")
    y_va = y[va_anchors]
    best_auc, best_state, bad = -1.0, None, 0
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time(); running = n_seen = 0
        for xb, lb, yb in tr_loader:
            xb = xb.to(device=device, dtype=torch.float32)
            lb = lb.to(device=device, dtype=torch.long)
            yb = yb.to(device=device, dtype=torch.float32)
            opt.zero_grad(set_to_none=True)
            logits = model(xb, lb).squeeze(-1)
            loss = loss_fn(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            opt.step()
            if sched is not None: sched.step()
            running += loss.item() * xb.size(0); n_seen += xb.size(0)
        va_auc = roc_auc_score(y_va, evaluate(model, va_loader, device))
        if verbose:
            print(f"      {name} ep{ep:>2}/{epochs} loss={running/max(n_seen,1):.4f} "
                  f"val_auc={va_auc:.4f} ({time.time()-t0:.1f}s)", flush=True)
        if va_auc > best_auc:
            best_auc = va_auc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= early_stop:
                if verbose: print(f"      early stop @ ep{ep}", flush=True)
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    te_pred = evaluate(model, te_loader, device)
    y_te = y[te_anchors]
    del model; gc.collect()
    if device.type == "cuda": torch.cuda.empty_cache()
    elif device.type == "mps": torch.mps.empty_cache()
    return {"model": name, "params": int(n_params), "val_auc": float(best_auc),
            "test_roc_auc": float(roc_auc_score(y_te, te_pred)),
            "test_pr_auc": float(average_precision_score(y_te, te_pred))}

def train_xgb(feat, y, tr_anchors, va_anchors, te_anchors, device, seed, n_estimators):
    Xtr, ytr = feat[tr_anchors].astype(np.float32), y[tr_anchors]
    Xva, yva = feat[va_anchors].astype(np.float32), y[va_anchors]
    Xte, yte = feat[te_anchors].astype(np.float32), y[te_anchors]
    pos = float((ytr == 1).sum()); neg = float((ytr == 0).sum())
    clf = xgb.XGBClassifier(
        n_estimators=n_estimators, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=2, reg_lambda=1.0,
        tree_method="hist", device="cuda" if device.type == "cuda" else "cpu",
        scale_pos_weight=max(neg / max(pos, 1.0), 1.0), eval_metric="auc",
        early_stopping_rounds=30, random_state=seed, n_jobs=-1)
    clf.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    p = clf.predict_proba(Xte)[:, 1]
    return {"model": "xgboost", "params": int(clf.best_iteration or n_estimators),
            "val_auc": float(roc_auc_score(yva, clf.predict_proba(Xva)[:, 1])),
            "test_roc_auc": float(roc_auc_score(yte, p)),
            "test_pr_auc": float(average_precision_score(yte, p))}

## 4 · Feature engineering (shared helpers + per-dataset adapters)

`add_time_gap_features` builds per-entity history aggregations from **past rows
only** (vectorized, so it scales to 24M rows). Frequency encoders and the
standardizer are fit on **train rows only**. `finalize` imputes, standardizes,
clips to ±5, sorts by `(entity, time)`, and returns the flat window cache.

In [7]:
def add_time_gap_features(df, ent_col, time_col, amt_col):
    grp = df[ent_col]
    g = df.groupby(ent_col, sort=False)
    df["delta_seconds_prev"] = g[time_col].diff().fillna(0).astype("float32")
    cnt_prev = g.cumcount()                                  # rows strictly before this one
    df["count_so_far"] = (cnt_prev + 1).astype("float32")
    prev_amt = g[amt_col].shift(1)
    df["prev_amt"] = prev_amt.fillna(-1).astype("float32")
    df["amt_diff_prev"] = (df[amt_col] - prev_amt).fillna(0).astype("float32")
    df["amt_ratio_prev"] = (df[amt_col] / prev_amt.replace(0, np.nan)).fillna(1.0).clip(0, 100).astype("float32")
    csum_prev = prev_amt.groupby(grp).cumsum()               # vectorized expanding mean/max over PAST
    df["amt_cummean"] = (csum_prev / cnt_prev.replace(0, np.nan)).fillna(-1).astype("float32")
    df["amt_cummax"]  = prev_amt.groupby(grp).cummax().fillna(-1).astype("float32")
    return ["delta_seconds_prev", "count_so_far", "prev_amt", "amt_diff_prev",
            "amt_ratio_prev", "amt_cummean", "amt_cummax"]

def freq_encode(df, cols, train_mask):
    out = []
    for c in cols:
        vc = df.loc[train_mask, c].value_counts(dropna=True)
        name = c + "_FE"
        df[name] = df[c].map(vc).fillna(0).astype("float32")
        out.append(name)
    return out

def add_cyclical(df, col, period):
    rad = 2 * np.pi * df[col].astype("float32") / period
    df[col + "_sin"] = np.sin(rad).astype("float32")
    df[col + "_cos"] = np.cos(rad).astype("float32")
    return [col + "_sin", col + "_cos"]

def finalize(df, feature_cols, train_mask, ent_int, time_int, y):
    """Impute, standardize (fit on train), clip, sort by (entity,time); return window cache."""
    feature_cols = list(dict.fromkeys(feature_cols))
    df[feature_cols] = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(-1).astype("float32")
    # Standardize every non-cyclical column (cyclical sin/cos already live in [-1,1]).
    # Standardizing binaries too is harmless and stops large FE counts from
    # overflowing float16 when a column is near-constant.
    to_scale = [c for c in feature_cols if not c.endswith(("_sin", "_cos"))]
    scaler = StandardScaler()
    df.loc[train_mask, to_scale]  = scaler.fit_transform(df.loc[train_mask, to_scale]).astype("float32")
    df.loc[~train_mask, to_scale] = scaler.transform(df.loc[~train_mask, to_scale]).astype("float32")
    df[to_scale] = df[to_scale].clip(-5.0, 5.0).astype("float32")
    order = np.lexsort((time_int, ent_int))                  # sort by entity, then time
    feat = df[feature_cols].to_numpy(dtype=SEQ_DTYPE)[order]
    ent_s = ent_int[order]
    y_s = y[order].astype(np.float32)
    split_s = df["__split__"].to_numpy()[order]
    boundaries = np.flatnonzero(np.r_[True, ent_s[1:] != ent_s[:-1]])
    seg_lens = np.diff(np.r_[boundaries, len(ent_s)])
    block_start = np.repeat(boundaries, seg_lens).astype(np.int64)
    assert np.isfinite(feat.astype(np.float32)).all(), "non-finite features remain"
    return feat, block_start, y_s, split_s, feature_cols

def pick_anchors(split_s, y_s, neg_per_pos, rng, full_test=True, train_cap=None, test_cap=None):
    """train/val: all positives + subsampled negatives; test: full (or capped)."""
    def subsample(idx):
        pos = idx[y_s[idx] == 1]; neg = idx[y_s[idx] == 0]
        k = min(len(neg), int(len(pos) * neg_per_pos)) if len(pos) else min(len(neg), 5000)
        neg = rng.choice(neg, size=k, replace=False) if k < len(neg) else neg
        a = np.concatenate([pos, neg]); rng.shuffle(a); return a
    tr = subsample(np.flatnonzero(split_s == 0))
    va = subsample(np.flatnonzero(split_s == 1))
    te = np.flatnonzero(split_s == 2)
    if not full_test:
        te = subsample(te)
    if train_cap is not None and len(tr) > train_cap:
        tr = rng.choice(tr, train_cap, replace=False)
    if test_cap is not None and len(te) > test_cap:
        te = rng.choice(te, test_cap, replace=False)
    return tr, va, te

### 4a · IBM TabFormer adapter (`card_transaction_v1.csv`)

In [8]:
def prepare_ibm(path, nrows=None):
    df = pd.read_csv(path, nrows=nrows)
    df.columns = [c.strip() for c in df.columns]
    ent_int = (df["User"].astype(np.int64) * 100 + df["Card"].astype(np.int64)).to_numpy()  # User+Card entity
    hm = df["Time"].str.split(":", expand=True)
    dt = pd.to_datetime(dict(year=df["Year"], month=df["Month"], day=df["Day"]), errors="coerce") \
         + pd.to_timedelta(hm[0].astype(int) * 3600 + hm[1].astype(int) * 60, unit="s")
    time_int = (dt.astype("int64") // 10**9).to_numpy()
    df["hour"] = hm[0].astype("float32")
    df["dow"]  = dt.dt.dayofweek.astype("float32")
    amt = df["Amount"].str.replace("$", "", regex=False).astype("float32")   # keep sign (refunds)
    df["amt"] = amt
    df["amt_log"] = (np.sign(amt) * np.log1p(np.abs(amt))).astype("float32")
    y = (df["Is Fraud?"].astype(str).str.strip() == "Yes").astype(np.int8).to_numpy()
    q = np.quantile(time_int, [0.70, 0.85])                                   # time-based 70/15/15
    split = np.where(time_int <= q[0], 0, np.where(time_int <= q[1], 1, 2)).astype(np.int8)
    df["__split__"] = split
    train_mask = split == 0
    df["Zip"] = df["Zip"].astype(str)
    cat_cols = ["Use Chip", "Merchant Name", "Merchant City", "Merchant State", "Zip", "MCC", "Errors?"]
    feat_cols  = ["amt_log"]
    feat_cols += freq_encode(df, cat_cols, train_mask)
    feat_cols += add_cyclical(df, "hour", 24)
    feat_cols += add_cyclical(df, "dow", 7)
    df["__ent__"] = ent_int; df["__t__"] = time_int
    feat_cols += add_time_gap_features(df, "__ent__", "__t__", "amt")
    return finalize(df, feat_cols, train_mask, ent_int, time_int, y)

### 4b · Sparkov adapter (`fraudTrain.csv` / `fraudTest.csv`)

In [9]:
def _haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1); dl = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return (2 * R * np.arcsin(np.sqrt(a))).astype("float32")

def prepare_sparkov(train_path, test_path, nrows=None):
    tr = pd.read_csv(train_path, nrows=nrows)
    te = pd.read_csv(test_path, nrows=(nrows // 2 if nrows else None))
    tr["__src__"] = 0; te["__src__"] = 2
    df = pd.concat([tr, te], ignore_index=True)                              # combine so a card's history bridges train->test
    ent_int = df["cc_num"].astype("category").cat.codes.to_numpy().astype(np.int64)
    time_int = df["unix_time"].astype(np.int64).to_numpy()
    dt = pd.to_datetime(df["trans_date_trans_time"], errors="coerce")
    df["hour"]  = dt.dt.hour.astype("float32")
    df["dow"]   = dt.dt.dayofweek.astype("float32")
    df["month"] = dt.dt.month.astype("float32")
    dob = pd.to_datetime(df["dob"], errors="coerce")
    df["age"] = ((dt - dob).dt.days / 365.25).astype("float32")
    df["amt_f"]        = df["amt"].astype("float32")
    df["amt_log"]      = np.log1p(df["amt"].clip(lower=0)).astype("float32")
    df["city_pop_log"] = np.log1p(df["city_pop"].clip(lower=0)).astype("float32")
    df["gender_bin"]   = (df["gender"].astype(str) == "M").astype("float32")
    df["distance"] = _haversine(df["lat"].to_numpy(), df["long"].to_numpy(),
                                df["merch_lat"].to_numpy(), df["merch_long"].to_numpy())
    y = df["is_fraud"].astype(np.int8).to_numpy()
    split = df["__src__"].to_numpy().copy()                                  # 0=train, 2=test
    tr_pos = np.flatnonzero(split == 0)
    cut = np.quantile(time_int[tr_pos], 0.90)                               # val = last 10% of train by time
    split[(split == 0) & (time_int > cut)] = 1
    df["__split__"] = split.astype(np.int8)
    train_mask = split == 0
    cat_cols = ["category", "merchant", "job", "state"]
    feat_cols  = ["amt_log", "city_pop_log", "gender_bin", "age", "distance",
                  "lat", "long", "merch_lat", "merch_long"]
    feat_cols += freq_encode(df, cat_cols, train_mask)
    feat_cols += add_cyclical(df, "hour", 24)
    feat_cols += add_cyclical(df, "dow", 7)
    feat_cols += add_cyclical(df, "month", 12)
    df["__ent__"] = ent_int; df["__t__"] = time_int
    feat_cols += add_time_gap_features(df, "__ent__", "__t__", "amt_f")
    return finalize(df, feat_cols, train_mask, ent_int, time_int, y)

## 5 · Run all four models on each dataset

In [ ]:
def run_dataset(tag, feat, block_start, y_s, split_s, feature_cols):
    rng = np.random.default_rng(SEED)
    tr, va, te = pick_anchors(split_s, y_s, NEG_PER_POS, rng,
                              full_test=not SMOKE, train_cap=TRAIN_CAP, test_cap=TEST_CAP)
    nF = feat.shape[1]
    print(f"\n##### {tag}: F={nF} | anchors train={len(tr):,} val={len(va):,} test={len(te):,} "
          f"| train_pos={int(y_s[tr].sum())} test_pos={int(y_s[te].sum())}", flush=True)
    rows = []
    for name in ["lstm_bidir", "cnn_temporal", "cnn_feature_lstm"]:
        print(f"  -- {name} --", flush=True)
        r = train_deep(name, nF, feat, block_start, y_s, tr, va, te, device,
                       EPOCHS, BATCH, LR, WEIGHT_DECAY, GRAD_CLIP, EARLY_STOP, SEED, NUM_WORKERS)
        r["dataset"] = tag; rows.append(r); print("    ", r, flush=True)
    print("  -- xgboost --", flush=True)
    r = train_xgb(feat, y_s, tr, va, te, device, SEED, XGB_TREES)
    r["dataset"] = tag; rows.append(r); print("    ", r, flush=True)
    return rows

all_results = []

if RUN_IBM:
    t0 = time.time()
    feat, bs, y_s, split_s, fc = prepare_ibm(IBM_CSV, nrows=IBM_NROWS)
    print(f"IBM prepared in {time.time()-t0:.1f}s  (rows={len(y_s):,}, F={feat.shape[1]})")
    all_results += run_dataset("IBM", feat, bs, y_s, split_s, fc)
    del feat, bs, y_s, split_s; gc.collect()

if RUN_SPARKOV:
    t0 = time.time()
    feat, bs, y_s, split_s, fc = prepare_sparkov(SPARKOV_TRAIN, SPARKOV_TEST, nrows=SPARKOV_NROWS)
    print(f"Sparkov prepared in {time.time()-t0:.1f}s  (rows={len(y_s):,}, F={feat.shape[1]})")
    all_results += run_dataset("Sparkov", feat, bs, y_s, split_s, fc)
    del feat, bs, y_s, split_s; gc.collect()

IBM prepared in 179.5s  (rows=24,386,900, F=19)

##### IBM: F=19 | anchors train=1,065,186 val=225,267 test=3,658,033 | train_pos=20886 test_pos=4454
  -- lstm_bidir --
      lstm_bidir ep 1/15 loss=0.1151 val_auc=0.9678 (59.1s)
      lstm_bidir ep 2/15 loss=0.0356 val_auc=0.9748 (66.9s)
      lstm_bidir ep 3/15 loss=0.0309 val_auc=0.9748 (63.9s)
      lstm_bidir ep 4/15 loss=0.0283 val_auc=0.9797 (65.1s)
      lstm_bidir ep 5/15 loss=0.0263 val_auc=0.9790 (64.6s)
      lstm_bidir ep 6/15 loss=0.0244 val_auc=0.9815 (64.3s)
      lstm_bidir ep 7/15 loss=0.0224 val_auc=0.9772 (64.7s)
      lstm_bidir ep 8/15 loss=0.0203 val_auc=0.9791 (65.1s)
      lstm_bidir ep 9/15 loss=0.0182 val_auc=0.9744 (64.6s)
      lstm_bidir ep10/15 loss=0.0162 val_auc=0.9720 (64.3s)
      lstm_bidir ep11/15 loss=0.0144 val_auc=0.9677 (64.2s)
      early stop @ ep11
     {'model': 'lstm_bidir', 'params': 631105, 'val_auc': 0.9814604593598661, 'test_roc_auc': 0.9654742442960443, 'test_pr_auc': 0.1985378292177188

## 6 · Comparison

In [ ]:
results_df = (pd.DataFrame(all_results)
              [["dataset", "model", "params", "val_auc", "test_roc_auc", "test_pr_auc"]]
              .sort_values(["dataset", "test_roc_auc"], ascending=[True, False])
              .reset_index(drop=True))
results_df.to_csv("model_comparison_results.csv", index=False)
print("Saved model_comparison_results.csv")
results_df

In [ ]:
# Grouped bar charts: test ROC-AUC and PR-AUC per dataset
datasets = list(results_df["dataset"].unique())
models = ["lstm_bidir", "cnn_temporal", "cnn_feature_lstm", "xgboost"]
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, metric, title in zip(axes, ["test_roc_auc", "test_pr_auc"], ["Test ROC-AUC", "Test PR-AUC"]):
    x = np.arange(len(datasets)); w = 0.2
    for k, m in enumerate(models):
        vals = [results_df[(results_df.dataset == d) & (results_df.model == m)][metric].mean() for d in datasets]
        bars = ax.bar(x + (k - 1.5) * w, vals, w, label=m)
        for b, v in zip(bars, vals):
            if not np.isnan(v):
                ax.text(b.get_x() + b.get_width()/2, v + 0.005, f"{v:.3f}",
                        ha="center", va="bottom", fontsize=7, rotation=90)
    ax.set_xticks(x); ax.set_xticklabels(datasets)
    ax.set_ylim(0, 1.05); ax.set_ylabel(metric); ax.set_title(title)
    ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.savefig("model_comparison.png", dpi=120, bbox_inches="tight"); plt.show()
print("Saved model_comparison.png")

### Notes
- **`test_roc_auc`** is threshold-free and prevalence-independent — the primary
  ranking metric. **`test_pr_auc`** (average precision) is more sensitive to the
  extreme class imbalance and is the better fraud-detection metric.
- Training keeps **all fraud + `NEG_PER_POS` legit per fraud**; the **test set is
  scored in full** (`TEST_CAP=None`), so PR-AUC reflects the true base rate.
- For a stronger result, raise `EPOCHS`, lower `NEG_PER_POS` (more negatives),
  and/or average several `SEED`s. XGBoost sees the same engineered features
  (incl. per-card history aggregations), making it a fair non-sequential baseline.